# Data Cleaning Guide: `area_harvested.xlsx`

This beginner-friendly notebook walks through cleaning **`area_harvested.xlsx`** step by step using **pandas**.

### Cleaning Checklist:
1. **Load & Inspect** the dataset.
2. **Remove Redundant Columns** (`Original_Location` and `Record_Type`).
3. **Reorder & Standardize Columns** (put `Location` first and format year headers as strings).
4. **Clean Text & Check Duplicates** in `Location`.
5. **Inspect & Handle Missing Values (`NaN`)** (`Maguindanao del Norte`).
6. **Validate Numbers & Save** the cleaned Excel file.

## Step 1: Import Pandas and Load the Dataset
We use `pd.read_excel()` to load the spreadsheet into a pandas DataFrame (a table).

In [1]:
import pandas as pd

# Load the Excel file
df = pd.read_excel('area_harvested.xlsx')

# Check how many rows and columns we have
print('Shape (rows, columns):', df.shape)

# Look at the first 5 rows
print(df.head())

Shape (rows, columns): (82, 42)
  Original_Location     1987     1988     1989     1990     1991     1992     1993     1994     1995     1996     1997     1998     1999     2000     2001     2002     2003     2004     2005     2006     2007     2008     2009     2010     2011     2012     2013     2014     2015     2016     2017     2018     2019      2020      2021      2022      2023      2024      2025 Record_Type Location
0              Abra  15870.0  16920.0  16580.0  16610.0  15370.0  14400.0  14520.0  14430.0  16750.0  16680.0  16460.0  15800.0  16062.0  16232.0  16272.0  16478.0  16612.0  16771.0  16800.0  16833.0  23615.0  22695.0  23675.0  23673.0  23882.0  24023.0  24031.0  24056.0  23987.0  23993.0  24014.0  23541.0  23542.0  23382.67  23383.41  21795.55  21795.99  21837.93  21838.00    Province     Abra
1            Apayao      0.0      0.0      0.0      0.0      0.0      0.0      0.0  11918.0  12742.0  12351.0  12389.0  12376.0   9999.0  18588.0  20323.0  20373.0  19232.0

## Step 2: Check for Redundant Columns
Notice that this file has both `Original_Location` and `Location`, plus a `Record_Type` column at the end. Let's check if they carry extra information or if they are duplicates.

In [2]:
# Check if Original_Location and Location are 100% identical
same_location = (df['Original_Location'] == df['Location']).all()
print('Are Original_Location and Location identical?:', same_location)

# Check what values exist inside Record_Type
print('Unique values in Record_Type:', df['Record_Type'].unique())

Are Original_Location and Location identical?: True
Unique values in Record_Type: ['Province']


## Step 3: Drop Redundant Columns & Reorder
Since `Original_Location` is an exact duplicate of `Location` and `Record_Type` is just `'Province'` for every row, we can drop both. Then we move `Location` to the first column so the table is easier to read.

In [3]:
# Drop the two redundant columns
df = df.drop(columns=['Original_Location', 'Record_Type'])

# Move 'Location' to the very front
year_cols = [col for col in df.columns if col != 'Location']
df = df[['Location'] + year_cols]

# Make sure all column names are strings
df.columns = df.columns.astype(str)

print('New column order (first 6):', list(df.columns[:6]))

New column order (first 6): ['Location', '1987', '1988', '1989', '1990', '1991']


## Step 4: Clean Province Names & Check Duplicates
Trimming extra spaces (`str.strip()`) prevents hidden mismatches if you later merge this table with another dataset.

In [4]:
# Remove any extra spaces before or after province names
df['Location'] = df['Location'].str.strip()

# Check if any province is listed more than once
print('Duplicate provinces:', df['Location'].duplicated().sum())

Duplicate provinces: 0


## Step 5: Inspect & Handle Missing Values (`NaN`)
Let's see which rows have missing values (`NaN`).

- **`Maguindanao del Norte`** has `NaN` from `1987` to `2023` because the province was newly created after the Maguindanao split.
- Other newly created provinces in this dataset (like `Apayao` and `Dinagat Islands`) use `0.0` for years before they existed.

Below, we fill `NaN` with `0` so the whole table follows one consistent rule. *(Tip: If you want to calculate historical averages without zeros pulling down the mean, comment out `df = df.fillna(0)` and keep `NaN` instead!)*

In [5]:
# Show rows that have at least one missing value
missing_rows = df[df.isnull().any(axis=1)]
print('Rows with missing values:')
print(missing_rows[['Location', '1987', '2023', '2024', '2025']])

# Replace NaN with 0
df = df.fillna(0)

print('Total missing values remaining:', df.isnull().sum().sum())

Rows with missing values:
                 Location  1987  2023      2024      2025
78  Maguindanao del Norte   NaN   NaN  70659.91  70659.91
Total missing values remaining: 0


## Step 6: Validate Numeric Values & Save
Let's verify there are no negative numbers, check summary statistics, and save the cleaned dataset to a new Excel file.

In [6]:
year_cols = [col for col in df.columns if col != 'Location']

# Ensure all year columns are numeric
for col in year_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Check for negative numbers
print('Negative values count:', (df[year_cols] < 0).sum().sum())

# Save the cleaned table to a new Excel file
df.to_excel('area_harvested_cleaned.xlsx', index=False)
print('Saved successfully to area_harvested_cleaned.xlsx!')

Negative values count: 0
Saved successfully to area_harvested_cleaned.xlsx!
